# Cấp độ 1 — Knowledge Tracing trên đồ thị (bản đầy đủ cho báo cáo)

Notebook này thay thế `kaggle_level1.ipynb` khi cần **đủ số liệu, đủ bằng chứng và đóng gói một lần**.
Notebook cũ chạy được nhưng còn các khoảng trống đã ghi trong `GAP_LEVEL1.md`:

| Khoảng trống (GAP) | Bản cũ | Notebook này |
|---|---|---|
| P0.1 đa seed, **split cố định**, có cả `Dense` | Thiếu `Dense`; chỉ lưu mean/std | §3 chạy **7 cấu hình × 5 seed** trên cùng split; lưu từng seed |
| P1.6 case study có bằng chứng tính toán | Chỉ lịch sử + láng giềng | §7 thêm **phản chứng lật kết quả**, **ablation đồ thị tại case**, **occlusion bỏ hẳn một bước**, **trạng thái tri thức h_t**, chọn case theo tầng confidence |
| P1.8 `Identity ≡ Zero`, thiếu edge-shuffle | Có `Shuffled` nhưng xáo cột toàn cục | §2 thêm `ShuffledRow` giữ **đúng out-degree từng hàng** |
| E3 chưa chứng minh vai trò **cấu trúc** đồ thị | Không có kiểm định | §4 **paired t-test + bootstrap CI + Cohen's d_z**; §6 **ablation A:=0 lúc suy luận** + độ phủ đồ thị |
| E2 mean ± std ≥ 3 seed | 3 seed | 5 seed (đổi được bằng `SEEDS`) |
| D4 dashboard chọn người học | Có code, JSON rất lớn | §8 giới hạn 25 người học, thêm bảng tóm tắt + hình một người học |
| D1/D2 artifact tái lập | Có split, thiếu manifest/leak report | §2 `leak_report.txt`; §9 `run_manifest.json` + `environment.txt` |
| P0.3/P2.11 sản phẩm cho báo cáo | Chỉ CSV/JSON rời | §9 `report_summary.md`; §10 `level1_output.zip` + `level1_report_pack.zip` |

> Chạy `QUICK_RUN=True` chỉ để kiểm tra mã (15 người học, 1 seed). Số liệu báo cáo phải chạy
> `QUICK_RUN=False` trên ASSISTments 2009 đầy đủ.


## 0. Clone repo và cấu hình môi trường

Notebook `git clone` repo rồi gọi package `level1/`. Trên Kaggle, mọi sửa đổi package phải
**commit + push** trước khi chạy; notebook này không cần sửa package, toàn bộ phân tích bổ sung
nằm trong cell để bản clone HEAD vẫn chạy được.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhnnh688/Knowledge-Tracing.git"
KAGGLE_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / "Knowledge-Tracing"
    if not (REPO / ".git").exists():
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
    else:
        try:
            subprocess.check_call(
                ["git", "-C", str(REPO), "fetch", "origin", "main", "--depth", "1"]
            )
            subprocess.check_call(
                ["git", "-C", str(REPO), "reset", "--hard", "origin/main"]
            )
        except Exception as exc:
            print(f"Git sync lỗi ({exc}) -> clone lại")
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
else:
    REPO = (
        Path.cwd()
        if (Path.cwd() / "level1").exists()
        else Path.cwd() / "Knowledge-Tracing"
    )

os.chdir(REPO)
GKT_DIR = REPO / "GKT"
for p in (REPO, GKT_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
    ).strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repo:", REPO.resolve())
print("GKT:", GKT_DIR.resolve())
print("Git commit:", GIT_COMMIT)

### 0.1 Kiểm tra gói — không cài lại pandas/numpy của Kaggle

`kaggle_level1.ipynb` cũ chạy `%pip install pandas scipy ...` vô điều kiện. Trên image mới
(pandas 3.x / numpy 2.x) thao tác đó có thể hạ cấp gói và làm hỏng môi trường. Cell dưới chỉ
cài gói **thật sự thiếu**, và dừng nếu thiếu PyTorch (Kaggle GPU image đã có sẵn).


In [ ]:
import importlib.util
import subprocess
import sys

MODULES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
}
missing = [pkg for mod, pkg in MODULES.items() if importlib.util.find_spec(mod) is None]
print("Gói thiếu:", missing if missing else "không")

if "torch" in missing:
    raise RuntimeError(
        "Không tìm thấy PyTorch. Trên Kaggle hãy bật Accelerator=GPU (image đã có torch); "
        "không pip install torch trong notebook để tránh cài nhầm bản CPU."
    )
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(
    "numpy",
    np.__version__,
    "| pandas",
    pd.__version__,
    "| scipy",
    scipy.__version__,
    "| sklearn",
    sklearn.__version__,
)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### 0.2 Cấu hình thực nghiệm

Các nút cần đổi khi làm báo cáo:

| Biến | Mặc định | Ý nghĩa |
|---|---|---|
| `QUICK_RUN` | `False` | `True` = 15 người học, 1 seed, chỉ kiểm tra mã |
| `SEEDS` | `[42, 123, 456, 789, 1011]` | 5 seed, split cố định seed 42 |
| `EPOCHS` | 20 | early stopping theo val AUC, patience 5 |
| `DASHBOARD_MAX_LEARNERS` | 25 | giới hạn JSON dashboard cho nhẹ |

Calibration (§5), ablation suy luận (§6) và case study (§7) luôn chạy vì rất nhẹ so với huấn luyện;
chỉ dashboard là có công tắc `RUN_DASHBOARD`.


In [ ]:
import gc
import hashlib
import json
import math
import platform
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from level1.dashboard import (
    export_learner_view,
    interactive_learner_view,
    render_learner_view,
)
from level1.explain import (
    annotate_evidence,
    skill_lookup_from_loader,
    user_ids_from_loader,
)
from level1.pipeline import (
    build_model,
    evaluate,
    load_kt_data,
    resolve_data_file,
    run_experiment,
    save_split,
    set_seed,
    train_one,
)

NOTEBOOK_VERSION = "2026-11-05 level1-full"

# True = smoke test 15 người học, KHÔNG dùng số liệu cho báo cáo.
QUICK_RUN = False

DATA_FILE = "assistment_test15.csv" if QUICK_RUN else "skill_builder_data.csv"
EPOCHS = 8 if QUICK_RUN else 20
BATCH_SIZE = 16 if QUICK_RUN else 32
MAX_SEQ_LEN = 80 if QUICK_RUN else 200
SEEDS = [42] if QUICK_RUN else [42, 123, 456, 789, 1011]

GRAPH_TYPES = ["Transition", "Dense", "Identity", "Zero", "Shuffled"]
MODEL_NAMES = ["DKT"] + [f"GKT-{g}" for g in GRAPH_TYPES] + ["GKT-ShuffledRow"]

RUN_DASHBOARD = True
DASHBOARD_MAX_LEARNERS = 15 if QUICK_RUN else 25

IS_KAGGLE = Path("/kaggle/working").exists()
SAVE_DIR = (
    Path("/kaggle/working/level1_out")
    if IS_KAGGLE
    else (REPO / "outputs" / "level1_full")
)
TABLES_DIR = SAVE_DIR / "tables"
FIG_DIR = SAVE_DIR / "figures"
CKPT_DIR = SAVE_DIR / "checkpoints"
for _d in (SAVE_DIR, TABLES_DIR, FIG_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

ZIP_PATH = (
    Path("/kaggle/working/level1_output.zip")
    if IS_KAGGLE
    else (REPO / "outputs" / "level1_output.zip")
)
REPORT_ZIP_PATH = ZIP_PATH.with_name("level1_report_pack.zip")

DATA_PATH = Path(resolve_data_file(DATA_FILE))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Notebook: {NOTEBOOK_VERSION}")
print(f"Data: {DATA_PATH}")
print(f"Device: {device} | seeds={SEEDS}")
print(f"Output: {SAVE_DIR}")


def save_fig(fig, name, dpi=150):
    path = FIG_DIR / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print("[fig]", path)
    return path


def md_table(df, float_fmt="{:.4f}"):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        vals = []
        for v in row:
            if isinstance(v, (float, np.floating)):
                vals.append("nan" if pd.isna(v) else float_fmt.format(float(v)))
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def write_csv(df, name):
    path = TABLES_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("[csv]", path)
    return path


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return str(obj)


def write_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=json_default),
        encoding="utf-8",
    )
    print("[json]", path)
    return path

## 1. Khám phá dữ liệu (EDA)

Ghi lại thống kê dữ liệu gốc, hash SHA-256 (để chứng minh tái lập), và phân phối độ dài chuỗi.
Hình được lưu thành PNG để dán vào báo cáo, không chỉ hiện inline.


In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
user_len = df_raw.groupby("user_id").size()

DATA_STATS = {
    "file": DATA_PATH.name,
    "sha256": sha256_file(DATA_PATH),
    "size_mb": round(DATA_PATH.stat().st_size / 1024**2, 2),
    "rows": int(len(df_raw)),
    "users": int(df_raw["user_id"].nunique()),
    "skills": int(df_raw["skill_id"].nunique(dropna=True)),
    "problems": (
        int(df_raw["problem_id"].nunique()) if "problem_id" in df_raw.columns else -1
    ),
    "correct_rate": float(df_raw["correct"].eq(1).mean()),
    "missing_skill_rows": int(df_raw["skill_id"].isna().sum()),
    "users_lt3": int((user_len < 3).sum()),
}

stats_df = pd.DataFrame([{"Chỉ số": k, "Giá trị": v} for k, v in DATA_STATS.items()])
write_csv(stats_df, "dataset_stats.csv")

print(
    f"Rows={DATA_STATS['rows']:,} | users={DATA_STATS['users']:,} | "
    f"skills={DATA_STATS['skills']:,} | correct={DATA_STATS['correct_rate']:.3%}"
)
print(user_len.describe(percentiles=[0.5, 0.9, 0.99]).to_string())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(user_len.to_numpy(), bins=60, edgecolor="black", alpha=0.75)
ax.set_xlabel("Số lượt tương tác / người học")
ax.set_ylabel("Số người học")
ax.set_title(f"Phân phối độ dài chuỗi — {DATA_FILE}")
ax.grid(alpha=0.25)
save_fig(fig, "eda_sequence_length.png")

## 2. Chia tập, xây đồ thị và kiểm tra chống rò rỉ

- Chia **theo người học**, `seed=42`, tỉ lệ `0.6/0.2/0.2`; mỗi người là một chuỗi.
- Mọi bundle (`Transition`, `Dense`, `Identity`, `Zero`, `Shuffled`) nạp với **cùng seed 42**,
  nên chỉ đồ thị thay đổi, split giữ nguyên.
- Đồ thị `Transition` chỉ đếm chuyển tiếp của **người học trong train**.
- Cell dưới kiểm tra ba thứ: split rời nhau, graph support khớp transition của train, và nhãn
  được dịch đúng 1 bước (`feature_t = 2*skill_t + correct_t`, dự đoán `t+1`).


In [ ]:
print(">>> Nạp dữ liệu với graph=Transition, split_seed=42 ...")
shared = load_kt_data(
    DATA_FILE,
    batch_size=BATCH_SIZE,
    graph_type="Transition",
    model_type="GKT",
    seed=42,
    max_seq_len=MAX_SEQ_LEN,
    use_cuda=device.type == "cuda",
)
split = save_split(shared, SAVE_DIR / "split.json")

train_users = set(split["train_users"])
val_users = set(split["val_users"])
test_users = set(split["test_users"])

assert train_users.isdisjoint(val_users), "train/val trùng người học"
assert train_users.isdisjoint(test_users), "train/test trùng người học"
assert val_users.isdisjoint(test_users), "val/test trùng người học"


def _loader_interactions(loader):
    ds = loader.dataset
    base = getattr(ds, "dataset", ds)
    indices = getattr(ds, "indices", range(len(base)))
    return int(sum(len(base.questions[i]) for i in indices))


split_df = pd.DataFrame(
    [
        {
            "Tập": role,
            "Số người học": len(users),
            "Số tương tác (sau cắt)": _loader_interactions(loader),
        }
        for role, loader, users in [
            ("train", shared["train_loader"], train_users),
            ("val", shared["val_loader"], val_users),
            ("test", shared["test_loader"], test_users),
        ]
    ]
)
write_csv(split_df, "split_summary.csv")
print(split_df.to_string(index=False))
print("Concept_num =", shared["concept_num"])
print("Split hợp lệ: ba tập rời nhau theo người học.")

### 2.1 Nạp các biến thể đồ thị trên cùng split

`Shuffled` của package xáo **toàn bộ cột đích** trên ma trận, nên out-degree từng hàng có thể đổi.
Để kiểm soát chặt hơn, notebook dựng thêm `ShuffledRow`: hoán vị đích **trong từng hàng**, giữ
nguyên số cạnh đi ra của mỗi khái niệm. Đây là biến thể phá cấu trúc nhưng khớp mật độ theo hàng.


In [ ]:
def row_degree_preserving_shuffle(graph, seed=42):
    """Hoán vị đích trong từng hàng, giữ nguyên out-degree của mỗi khái niệm."""
    g = graph.detach().cpu().numpy().astype("float64").copy()
    rng = np.random.RandomState(seed)
    out = np.zeros_like(g)
    for i in range(g.shape[0]):
        cols = np.nonzero(g[i])[0]
        if cols.size == 0:
            continue
        out[i, rng.permutation(cols)] = g[i, cols]
    np.fill_diagonal(out, 0.0)
    rowsum = out.sum(1, keepdims=True)
    rowsum[rowsum == 0] = 1.0
    return torch.from_numpy(out / rowsum).float()


print(">>> Nạp các bundle đồ thị với CÙNG split seed=42 ...")
GRAPH_BUNDLES = {"Transition": shared}
for g in ["Dense", "Identity", "Zero", "Shuffled"]:
    print(f"--- graph={g} ---")
    GRAPH_BUNDLES[g] = load_kt_data(
        DATA_FILE,
        batch_size=BATCH_SIZE,
        graph_type=g,
        model_type="GKT",
        seed=42,
        max_seq_len=MAX_SEQ_LEN,
        use_cuda=device.type == "cuda",
    )


def _split_user_set(loader):
    return set(user_ids_from_loader(loader))


ref_split = {
    role: _split_user_set(shared[f"{role}_loader"]) for role in ("train", "val", "test")
}
for g, b in GRAPH_BUNDLES.items():
    for role in ("train", "val", "test"):
        got = _split_user_set(b[f"{role}_loader"])
        assert got == ref_split[role], f"{g}: split {role} lệch so với Transition"
print("Split giống hệt nhau trên mọi bundle: OK")

ROW_SHUFFLED_GRAPH = row_degree_preserving_shuffle(
    GRAPH_BUNDLES["Transition"]["graph"], seed=42
)


def graph_stats(name, graph):
    g = graph.detach().cpu().numpy()
    k = g.shape[0]
    nz = g != 0
    return {
        "Đồ thị": name,
        "Số đỉnh": int(k),
        "Số cạnh khác 0": int(nz.sum()),
        "Mật độ": float(nz.sum() / max(1, k * (k - 1))),
        "Số hàng có cạnh": int((nz.sum(1) > 0).sum()),
        "Số cột có cạnh": int((nz.sum(0) > 0).sum()),
        "Tổng trọng số hàng TB": float(g.sum(1).mean()),
        "Trọng số lớn nhất": float(g.max()) if g.size else 0.0,
        "Đường chéo khác 0": int(np.count_nonzero(np.diag(g))),
    }


G_TRANS = GRAPH_BUNDLES["Transition"]["graph"].detach().cpu()
G_SHUF = GRAPH_BUNDLES["Shuffled"]["graph"].detach().cpu()

outdeg_trans = (G_TRANS.numpy() != 0).sum(1)
outdeg_shuf = (G_SHUF.numpy() != 0).sum(1)
outdeg_shufrow = (ROW_SHUFFLED_GRAPH.numpy() != 0).sum(1)

graph_stats_df = pd.DataFrame(
    [graph_stats(name, b["graph"]) for name, b in GRAPH_BUNDLES.items()]
    + [graph_stats("ShuffledRow", ROW_SHUFFLED_GRAPH)]
)
write_csv(graph_stats_df, "graph_stats.csv")
print(graph_stats_df.to_string(index=False))
print(
    "Out-degree giữ nguyên theo hàng: ShuffledRow =",
    bool((outdeg_shufrow == outdeg_trans).all()),
    "| Shuffled toàn cục =",
    bool((outdeg_shuf == outdeg_trans).all()),
)

### 2.2 Độ phủ đồ thị và báo cáo chống rò rỉ

Hai câu hỏi cần trả lời bằng số, vì chúng giải thích kết quả Cấp độ 1:

1. **Bao nhiêu dự đoán test có khái niệm đích thực sự có cạnh** trong đồ thị train-only? Nếu phần
   lớn khái niệm không có cạnh đi ra, GKT gần như không có tín hiệu đồ thị để lan truyền.
2. **Đồ thị có chứa chuyển tiếp của người học ngoài train không?** Câu trả lời phải là không.
   Cell dưới đối chiếu support của đồ thị với transition đếm từ train, và đo khối lượng chuyển
   tiếp chỉ đến từ người học ngoài train.


In [ ]:
test_base = shared["test_loader"].dataset.dataset
test_indices = list(shared["test_loader"].dataset.indices)
G_np = G_TRANS.numpy()
outdeg = (G_np != 0).sum(1)
indeg = (G_np != 0).sum(0)

skill_counts = Counter()
for i in test_indices:
    qs = test_base.questions[i]
    for sid in qs[1:]:
        if sid >= 0:
            skill_counts[int(sid)] += 1
total_test_predictions = int(sum(skill_counts.values()))

coverage_rows = []
for label, mask in [
    ("Có cạnh đi ra (out-degree > 0)", outdeg > 0),
    ("Có cạnh đi vào (in-degree > 0)", indeg > 0),
    ("Có ít nhất một cạnh (out hoặc in)", (outdeg > 0) | (indeg > 0)),
]:
    covered = int(
        sum(c for s, c in skill_counts.items() if s < len(mask) and bool(mask[s]))
    )
    coverage_rows.append(
        {
            "Tiêu chí": label,
            "Số dự đoán test": total_test_predictions,
            "Số dự đoán được phủ": covered,
            "Tỉ lệ": covered / max(1, total_test_predictions),
        }
    )
coverage_df = pd.DataFrame(coverage_rows)
write_csv(coverage_df, "graph_coverage.csv")
print(coverage_df.to_string(index=False))

skill_lookup = skill_lookup_from_loader(shared["test_loader"])
skill_degree_df = pd.DataFrame(
    [
        {
            "skill": s,
            "skill_name": skill_lookup.get(s, str(s)),
            "out_degree": int(outdeg[s]),
            "in_degree": int(indeg[s]),
            "test_targets": int(skill_counts.get(s, 0)),
        }
        for s in range(len(outdeg))
    ]
)
write_csv(skill_degree_df, "graph_degree_by_skill.csv")


def transition_counts(base, indices, concept_num):
    counts = np.zeros((concept_num, concept_num), dtype=np.int64)
    for i in indices:
        qs = base.questions[i]
        for a, b in zip(qs[:-1], qs[1:]):
            if a >= 0 and b >= 0 and a != b:
                counts[a, b] += 1
    return counts


train_base = shared["train_loader"].dataset.dataset
train_indices = list(shared["train_loader"].dataset.indices)
c_train = transition_counts(train_base, train_indices, shared["concept_num"])
c_all = transition_counts(train_base, range(len(train_base)), shared["concept_num"])
c_heldout = c_all - c_train

support_train = c_train > 0
support_graph = G_np > 0
support_match = bool((support_train == support_graph).all())
rowsum = G_np.sum(1)
normalized_ok = bool(np.allclose(rowsum[rowsum > 0], 1.0, atol=1e-5))

test_only_mask = (c_all > 0) & (c_train == 0)
mass_all = int(c_all.sum())
mass_test_only = int(c_all[test_only_mask].sum())
mass_heldout_total = int(c_heldout.sum())
mass_heldout_on_train_edges = int(c_heldout[support_train].sum())

rng = np.random.RandomState(0)
sample_idx = rng.choice(len(train_base), size=min(200, len(train_base)), replace=False)
label_align_ok = True
bad_example = None
for i in sample_idx:
    feats = train_base.features[i]
    qs = train_base.questions[i]
    ans = train_base.answers[i]
    for t, sid in enumerate(qs):
        if sid < 0:
            continue
        if int(feats[t]) != 2 * int(sid) + int(ans[t]):
            label_align_ok = False
            bad_example = (int(i), int(t), int(feats[t]), int(sid), int(ans[t]))
            break
    if not label_align_ok:
        break

leak_lines = [
    "BÁO CÁO CHỐNG RÒ RỈ DỮ LIỆU — CẤP ĐỘ 1",
    f"Thời điểm: {utc_now()}",
    f"Dữ liệu: {DATA_PATH.name} | sha256={DATA_STATS['sha256']}",
    f"Git commit: {GIT_COMMIT}",
    "",
    "1) Chia tập theo người học (seed=42, 0.6/0.2/0.2)",
    f"   train/val/test = {len(train_users)}/{len(val_users)}/{len(test_users)} người học",
    "   giao train∩val = train∩test = val∩test = 0 (đã assert).",
    "",
    "2) Đồ thị Transition chỉ xây từ tập train",
    f"   support(counts_train > 0) == support(graph > 0): {support_match}",
    f"   số cạnh (đã bỏ self-loop): {int(support_graph.sum())}",
    f"   row-normalized (mỗi hàng khác 0 có tổng = 1): {normalized_ok}",
    f"   tổng khối lượng chuyển tiếp toàn dữ liệu (bỏ self-loop): {mass_all}",
    f"   khối lượng chỉ đến từ người học ngoài train (val+test): {mass_test_only} "
    f"({mass_test_only / max(1, mass_all):.2%} toàn bộ) — các cạnh này KHÔNG có trong đồ thị.",
    f"   khối lượng chuyển tiếp của người ngoài train rơi vào cạnh có trong train: "
    f"{mass_heldout_on_train_edges}/{mass_heldout_total} "
    f"({mass_heldout_on_train_edges / max(1, mass_heldout_total):.1%}).",
    "",
    "3) Nhãn dịch đúng 1 bước",
    f"   kiểm tra ngẫu nhiên {len(sample_idx)} chuỗi: feature_t = 2*skill_t + correct_t -> {label_align_ok}"
    + ("" if label_align_ok else f" (lỗi tại {bad_example})"),
    "   mô hình dự đoán vị trí t+1 từ input đến t; không dùng nhãn tương lai.",
    "",
    "4) Test loader không shuffle (shuffle=False trong GKT/processing.py) -> vị trí case khớp dữ liệu.",
    "",
    "KẾT LUẬN: Không có người học nào xuất hiện ở hai tập; đồ thị không chứa chuyển tiếp "
    "chỉ quan sát được ở val/test; nhãn không bị dùng làm đặc trưng.",
]
leak_text = "\n".join(leak_lines)
(SAVE_DIR / "leak_report.txt").write_text(leak_text, encoding="utf-8")
print(leak_text)

## 3. Huấn luyện: 7 cấu hình × 5 seed trên cùng split

Mỗi cấu hình dùng **cùng train/val/test** (seed split 42). Chỉ seed huấn luyện đổi (khởi tạo
trọng số + thứ tự batch), nên `mean ± std` phản ánh **phương sai huấn luyện**, không lẫn phương
sai split. Checkpoint chỉ lưu cho seed 42 để zip nhẹ; số liệu của mọi seed vẫn được lưu.

`GKT-ShuffledRow` được huấn luyện bằng `build_model` + `train_one` + `evaluate` vì bundle chuẩn
không có loại đồ thị này; mọi thứ còn lại giống hệt `run_experiment`.


In [ ]:
def run_custom_graph_experiment(
    name,
    graph,
    bundle,
    seed,
    epochs,
    lr=1e-3,
    hid_dim=32,
    emb_dim=32,
    save_dir=None,
    device=None,
):
    set_seed(seed)
    model = build_model(
        "GKT",
        bundle["concept_num"],
        graph=graph,
        graph_type="Transition",
        hid_dim=hid_dim,
        emb_dim=emb_dim,
        dropout=0.0,
        device=device,
    )
    history, best_val_auc = train_one(
        model,
        bundle["train_loader"],
        bundle["val_loader"],
        model_type="GKT",
        epochs=epochs,
        lr=lr,
        device=device,
    )
    test_metrics = evaluate(
        model, bundle["test_loader"], model_type="GKT", device=device
    )
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), save_dir / f"{name}.pt")
    result = {
        "name": name,
        "model_type": "GKT",
        "graph_type": name.replace("GKT-", ""),
        "concept_num": bundle["concept_num"],
        "best_val_auc": best_val_auc,
        "test_loss": test_metrics["loss"],
        "test_auc": test_metrics["auc"],
        "test_acc": test_metrics["acc"],
        "history": history,
        "seed": seed,
        "model": model.cpu(),
        "bundle": bundle,
    }
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(
        f">>> {name} test AUC={result['test_auc']:.4f} "
        f"Acc={result['test_acc']:.4f} Loss={result['test_loss']:.4f}"
    )
    return result


CONFIGS = [{"name": "DKT", "model_type": "DKT", "graph_type": "N/A", "bundle": shared}]
for g in GRAPH_TYPES:
    CONFIGS.append(
        {
            "name": f"GKT-{g}",
            "model_type": "GKT",
            "graph_type": g,
            "bundle": GRAPH_BUNDLES[g],
        }
    )
CONFIGS.append(
    {
        "name": "GKT-ShuffledRow",
        "model_type": "custom",
        "graph_type": "ShuffledRow",
        "bundle": shared,
        "graph": ROW_SHUFFLED_GRAPH,
    }
)

runs = []
seed_models = {}
t_start = time.time()
print(f">>> Huấn luyện {len(CONFIGS)} cấu hình × {len(SEEDS)} seed ...")

for cfg in CONFIGS:
    for s in SEEDS:
        t0 = time.time()
        if cfg["model_type"] == "custom":
            res = run_custom_graph_experiment(
                cfg["name"],
                cfg["graph"],
                cfg["bundle"],
                s,
                EPOCHS,
                save_dir=(CKPT_DIR if s == 42 else None),
                device=device,
            )
        else:
            res = run_experiment(
                DATA_FILE,
                model_type=cfg["model_type"],
                graph_type=cfg["graph_type"],
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                lr=1e-3,
                hid_dim=32,
                emb_dim=32,
                seed=s,
                max_seq_len=MAX_SEQ_LEN,
                save_dir=(CKPT_DIR if s == 42 else None),
                bundle=cfg["bundle"],
                device=device,
            )
        elapsed = time.time() - t0
        n_params = int(
            sum(p.numel() for p in res["model"].parameters() if p.requires_grad)
        )
        runs.append(
            {
                "model": cfg["name"],
                "graph": cfg["graph_type"],
                "seed": s,
                "test_auc": float(res["test_auc"]),
                "test_acc": float(res["test_acc"]),
                "test_loss": float(res["test_loss"]),
                "best_val_auc": float(res["best_val_auc"]),
                "elapsed_s": elapsed,
                "n_params": n_params,
            }
        )
        if s == 42:
            seed_models[cfg["name"]] = res
        else:
            del res
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(
            f"  [{cfg['name']} seed={s}] AUC={runs[-1]['test_auc']:.4f} ({elapsed:.1f}s)"
        )

runs_df = pd.DataFrame(runs)
write_csv(runs_df, "runs_all_seeds.csv")
print(f"Tổng thời gian huấn luyện: {(time.time() - t_start) / 60:.1f} phút")

## 4. Kết quả, đa seed và kiểm định thống kê

Bảng chính lấy seed 42; bảng đa seed lấy trung bình trên 5 seed cùng split. Kiểm định dùng
**paired t-test** (ghép theo seed), **bootstrap CI 95%** cho chênh lệch trung bình, và **Cohen's
d_z**. Với n=5, p-value chỉ là một phần của câu chuyện; CI và effect size được báo kèm.


In [ ]:
main_table = runs_df[runs_df["seed"] == 42].copy()
_order = {name: i for i, name in enumerate(MODEL_NAMES)}
main_table["_order"] = main_table["model"].map(_order)
main_table = (
    main_table.sort_values("_order").drop(columns="_order").reset_index(drop=True)
)

main_display = main_table.rename(
    columns={
        "model": "Mô hình",
        "graph": "Loại đồ thị",
        "test_auc": "Test AUC",
        "test_acc": "Test Accuracy",
        "test_loss": "Test NLL",
        "best_val_auc": "Best Val AUC",
        "n_params": "Số tham số",
        "elapsed_s": "Thời gian (s)",
    }
)[
    [
        "Mô hình",
        "Loại đồ thị",
        "Test AUC",
        "Test Accuracy",
        "Test NLL",
        "Best Val AUC",
        "Số tham số",
        "Thời gian (s)",
    ]
]
write_csv(main_display, "level1_metrics.csv")
write_json(main_display.to_dict(orient="records"), TABLES_DIR / "metrics.json")
print("=== BẢNG CHÍNH (seed 42) ===")
print(main_display.to_string(index=False))

ms_rows = []
for name in MODEL_NAMES:
    sub = runs_df[runs_df["model"] == name].sort_values("seed")
    ms_rows.append(
        {
            "model": name,
            "graph": sub["graph"].iloc[0],
            "n_seeds": len(sub),
            "auc_mean": sub["test_auc"].mean(),
            "auc_std": sub["test_auc"].std(ddof=0),
            "acc_mean": sub["test_acc"].mean(),
            "acc_std": sub["test_acc"].std(ddof=0),
            "nll_mean": sub["test_loss"].mean(),
            "nll_std": sub["test_loss"].std(ddof=0),
        }
    )
ms_table = pd.DataFrame(ms_rows)
ms_display = ms_table.rename(
    columns={
        "model": "Mô hình",
        "graph": "Đồ thị",
        "n_seeds": "Số seed",
        "auc_mean": "AUC mean",
        "auc_std": "AUC std",
        "acc_mean": "Acc mean",
        "acc_std": "Acc std",
        "nll_mean": "NLL mean",
        "nll_std": "NLL std",
    }
)
write_csv(ms_display, "multiseed_metrics.csv")
print("\n=== ĐA SEED (mean ± std, split cố định seed=42) ===")
for _, r in ms_table.iterrows():
    print(
        f"{r['model']:18s} AUC {r['auc_mean']:.4f} ± {r['auc_std']:.4f} | "
        f"Acc {r['acc_mean']:.4f} ± {r['acc_std']:.4f} | "
        f"NLL {r['nll_mean']:.4f} ± {r['nll_std']:.4f}"
    )

from scipy import stats


def paired_test(df, a, b, metric="test_auc", n_boot=5000, seed=42):
    xa = df[df["model"] == a].sort_values("seed")[metric].to_numpy(dtype=float)
    xb = df[df["model"] == b].sort_values("seed")[metric].to_numpy(dtype=float)
    if len(xa) != len(xb) or len(xa) < 2:
        return None
    d = xa - xb
    if np.allclose(d, 0):
        t_stat, p_val = 0.0, 1.0
    else:
        t_stat, p_val = stats.ttest_rel(xa, xb)
    se = d.std(ddof=1) / math.sqrt(len(d))
    if se > 0:
        half = stats.t.ppf(0.975, len(d) - 1) * se
        ci_low, ci_high = d.mean() - half, d.mean() + half
    else:
        ci_low = ci_high = d.mean()
    sd = d.std(ddof=1)
    dz = d.mean() / sd if sd > 0 else float("nan")
    rng = np.random.default_rng(seed)
    boots = np.array(
        [rng.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)]
    )
    return {
        "comparison": f"{a} − {b}",
        "metric": metric,
        "n_seeds": len(d),
        "mean_a": float(xa.mean()),
        "mean_b": float(xb.mean()),
        "mean_diff": float(d.mean()),
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
        "boot_ci_low": float(np.percentile(boots, 2.5)),
        "boot_ci_high": float(np.percentile(boots, 97.5)),
        "t": float(t_stat),
        "p": float(p_val),
        "cohen_dz": float(dz),
        "significant_0.05": bool(p_val < 0.05),
    }


PAIRS = [
    ("DKT", "GKT-Transition"),
    ("GKT-Transition", "GKT-Zero"),
    ("GKT-Transition", "GKT-Identity"),
    ("GKT-Transition", "GKT-Dense"),
    ("GKT-Transition", "GKT-Shuffled"),
    ("GKT-Transition", "GKT-ShuffledRow"),
    ("GKT-Dense", "GKT-ShuffledRow"),
    ("GKT-Dense", "GKT-Zero"),
    ("GKT-Shuffled", "GKT-Zero"),
    ("GKT-ShuffledRow", "GKT-Zero"),
    ("GKT-Identity", "GKT-Zero"),
]
paired_rows = []
for a, b in PAIRS:
    for metric in ("test_auc", "test_loss"):
        row = paired_test(runs_df, a, b, metric=metric)
        if row is not None:
            paired_rows.append(row)
pair_cols = [
    "comparison",
    "metric",
    "n_seeds",
    "mean_a",
    "mean_b",
    "mean_diff",
    "ci95_low",
    "ci95_high",
    "boot_ci_low",
    "boot_ci_high",
    "t",
    "p",
    "cohen_dz",
    "significant_0.05",
]
# QUICK_RUN chỉ có 1 seed -> không ghép cặp được; giữ khung cột để cell sau vẫn chạy.
paired_df = (
    pd.DataFrame(paired_rows) if paired_rows else pd.DataFrame(columns=pair_cols)
)
write_csv(paired_df, "paired_tests.csv")
print("\n=== KIỂM ĐỊNH GHÉP CẶP THEO SEED ===")
for _, r in paired_df.iterrows():
    sig = "có ý nghĩa" if r["significant_0.05"] else "chưa đủ bằng chứng"
    print(
        f"{r['comparison']:36s} {r['metric']:10s} "
        f"Δ={r['mean_diff']:+.4f} CI95=[{r['ci95_low']:+.4f},{r['ci95_high']:+.4f}] "
        f"p={r['p']:.3f} dz={r['cohen_dz']:+.2f} -> {sig}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for name in MODEL_NAMES:
    hist = seed_models[name]["history"]
    axes[0].plot(hist["val_loss"], label=name, linewidth=1.3)
    axes[1].plot(hist["val_auc"], label=name, linewidth=1.3)
axes[0].set_title("Validation NLL theo epoch (seed 42)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("NLL")
axes[0].grid(alpha=0.25)
axes[1].set_title("Validation AUC theo epoch (seed 42)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=7)
fig.tight_layout()
save_fig(fig, "learning_curves.png")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
x = np.arange(len(ms_table))
axes[0].bar(x, ms_table["auc_mean"], yerr=ms_table["auc_std"], capsize=3, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(ms_table["model"], rotation=25, ha="right")
axes[0].set_ylabel("Test AUC")
axes[0].set_title(f"Test AUC — mean ± std trên {len(SEEDS)} seed")
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(x, ms_table["nll_mean"], yerr=ms_table["nll_std"], capsize=3, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ms_table["model"], rotation=25, ha="right")
axes[1].set_ylabel("Test NLL")
axes[1].set_title(f"Test NLL — mean ± std trên {len(SEEDS)} seed")
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
save_fig(fig, "model_comparison.png")

## 5. Hiệu chỉnh xác suất (mục mở rộng §7): Brier, ECE, ACE

AUC chỉ đo thứ hạng; một mô hình có thể xếp hạng tốt nhưng xác suất lệch. Phần này tính:

- **Brier** = trung bình `(p − y)²`.
- **ECE** = trung bình có trọng số `|p_bin − y_bin|` trên 10 ngăn đều.
- **ACE** = trung bình không trọng số của cùng khoảng lệch.

Dùng đúng mặt nạ với AUC/NLL (chỉ vị trí có nhãn) để so sánh được giữa các mô hình.


In [ ]:
from sklearn.metrics import log_loss, roc_auc_score


def collect_predictions(model, loader, model_type="GKT", device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            features, questions, answers = [x.to(device) for x in batch]
            out = model(features, questions)
            pred = out[0] if isinstance(out, tuple) else out
            real = answers[:, 1:]
            mask = (real != -1) & (questions[:, 1:] != -1)
            ys.append(real[mask].detach().cpu().numpy())
            ps.append(pred[mask].detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


def ece_ace_brier(y, p, n_bins=10):
    y = np.asarray(y, dtype="float64")
    p = np.asarray(p, dtype="float64")
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    edges[-1] += 1e-9
    bin_idx = np.clip(np.digitize(p, edges[1:-1], right=False), 0, n_bins - 1)
    ece, ace_terms, bins = 0.0, [], []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        avg_p = float(p[mask].mean())
        avg_y = float(y[mask].mean())
        gap = abs(avg_p - avg_y)
        ece += (n / len(p)) * gap
        ace_terms.append(gap)
        bins.append(
            {
                "bin": b,
                "lo": float(edges[b]),
                "hi": float(edges[b + 1]),
                "n": n,
                "avg_pred": avg_p,
                "avg_label": avg_y,
                "gap": gap,
            }
        )
    return {
        "ece": float(ece),
        "ace": float(np.mean(ace_terms)) if ace_terms else float("nan"),
        "bins": bins,
        "mean_pred": float(p.mean()),
        "observed_rate": float(y.mean()),
    }


calib_entries = {}
calib_rows, calib_bin_rows = [], []
for name in MODEL_NAMES:
    if name not in seed_models:
        continue
    res = seed_models[name]
    model = res["model"].to(device)
    y, p = collect_predictions(model, res["bundle"]["test_loader"], name, device)
    cal = ece_ace_brier(y, p, n_bins=10)
    calib_entries[name] = (p, y)
    calib_rows.append(
        {
            "model": name,
            "n": int(len(y)),
            "auc": float(roc_auc_score(y, p)),
            "nll": float(log_loss(y, p)),
            "brier": float(np.mean((p - y) ** 2)),
            "ece": cal["ece"],
            "ace": cal["ace"],
            "mean_pred": cal["mean_pred"],
            "observed_rate": cal["observed_rate"],
            "over_prediction": cal["mean_pred"] - cal["observed_rate"],
        }
    )
    for b in cal["bins"]:
        calib_bin_rows.append({"model": name, **b})
    model.cpu()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

calib_df = pd.DataFrame(calib_rows)
write_csv(calib_df, "calibration.csv")
write_csv(pd.DataFrame(calib_bin_rows), "calibration_bins.csv")
print("=== HIỆU CHỈNH (seed 42) ===")
print(calib_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 5.0))
ax.plot([0, 1], [0, 1], "k--", linewidth=1.0, label="hoàn hảo")
for name in ["DKT", "GKT-Transition", "GKT-Dense", "GKT-Zero", "GKT-ShuffledRow"]:
    if name not in calib_entries:
        continue
    p, y = calib_entries[name]
    cal = ece_ace_brier(y, p, n_bins=10)
    xs = [b["avg_pred"] for b in cal["bins"]]
    ys = [b["avg_label"] for b in cal["bins"]]
    ns = [b["n"] for b in cal["bins"]]
    ax.plot(xs, ys, linewidth=1.3, label=f"{name} (ECE={cal['ece']:.3f})")
    ax.scatter(xs, ys, s=np.clip(np.sqrt(ns) / 8.0, 4, 40), zorder=3)
ax.set_xlabel("xác suất dự đoán trung bình theo ngăn")
ax.set_ylabel("tỉ lệ làm đúng quan sát được")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Reliability diagram — test set (seed 42)", fontsize=11)
ax.grid(alpha=0.25)
ax.legend(fontsize=7)
fig.tight_layout()
save_fig(fig, "calibration_reliability.png")

## 6. Đồ thị đóng góp bao nhiêu **lúc suy luận**?

Huấn luyện với hai đồ thị khác nhau mà AUC bằng nhau chưa chứng minh mô hình *dùng* đồ thị.
Phép thử trực tiếp: giữ nguyên trọng số đã học, đặt `A := 0` khi suy luận rồi đo:

- `ΔAUC = AUC(A:=0) − AUC(A gốc)` — âm nghĩa là đồ thị giúp.
- `ΔNLL = NLL(A:=0) − NLL(A gốc)` — dương nghĩa là đồ thị giúp.
- `mean |Δp|` — mức thay đổi xác suất trung bình của từng dự đoán.

`GKT-Identity` và `GKT-Zero` phải cho `Δ = 0` (kiểm tra cài đặt). Kết quả này trả lời trực tiếp
câu hỏi E3 của `GAP_LEVEL1.md` và giải thích vì sao `Transition ≈ Zero` trên ASSISTments 2009.


In [ ]:
def predict_with_graph(model, loader, device, zero_graph=False):
    old = None
    if zero_graph and isinstance(getattr(model, "graph", None), torch.nn.Parameter):
        old = model.graph.data
        model.graph.data = torch.zeros_like(old)
    try:
        return collect_predictions(model, loader, "GKT", device)
    finally:
        if old is not None:
            model.graph.data = old


graph_inf_rows = []
for name in [f"GKT-{g}" for g in GRAPH_TYPES] + ["GKT-ShuffledRow"]:
    res = seed_models[name]
    model = res["model"].to(device)
    loader = res["bundle"]["test_loader"]
    y, p = predict_with_graph(model, loader, device, zero_graph=False)
    y0, p0 = predict_with_graph(model, loader, device, zero_graph=True)
    auc, auc0 = roc_auc_score(y, p), roc_auc_score(y0, p0)
    nll, nll0 = log_loss(y, p), log_loss(y0, p0)
    graph_inf_rows.append(
        {
            "model": name,
            "auc_with_graph": float(auc),
            "auc_zero_graph": float(auc0),
            "delta_auc": float(auc0 - auc),
            "nll_with_graph": float(nll),
            "nll_zero_graph": float(nll0),
            "delta_nll": float(nll0 - nll),
            "mean_abs_dp": float(np.mean(np.abs(p0 - p))),
            "max_abs_dp": float(np.max(np.abs(p0 - p))),
        }
    )
    model.cpu()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

graph_inf_df = pd.DataFrame(graph_inf_rows)
write_csv(graph_inf_df, "graph_inference_ablation.csv")
print("=== ABLATION ĐỒ THỊ LÚC SUY LUẬN (seed 42) ===")
print(graph_inf_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(graph_inf_df))
axes[0].bar(x, graph_inf_df["delta_auc"], alpha=0.85)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(graph_inf_df["model"], rotation=25, ha="right")
axes[0].set_ylabel("ΔAUC khi A := 0")
axes[0].set_title("Đồ thị giúp (ΔAUC < 0) hay không?")
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(x, graph_inf_df["mean_abs_dp"], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(graph_inf_df["model"], rotation=25, ha="right")
axes[1].set_ylabel("mean |Δp|")
axes[1].set_title("Mức thay đổi xác suất trung bình khi bỏ đồ thị")
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
save_fig(fig, "graph_inference_ablation.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(coverage_df["Tiêu chí"], coverage_df["Tỉ lệ"], alpha=0.85)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Tỉ lệ dự đoán test")
axes[0].set_title("Độ phủ đồ thị Transition (train only)")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.25)
weights = [skill_counts.get(s, 0) for s in range(len(outdeg))]
axes[1].hist(outdeg, bins=20, weights=weights, edgecolor="black", alpha=0.75)
axes[1].set_xlabel("out-degree của khái niệm đích")
axes[1].set_ylabel("số dự đoán test (có trọng số)")
axes[1].set_title("Phân phối bậc đi ra, tính theo tần suất dự đoán")
axes[1].grid(alpha=0.25)
fig.tight_layout()
save_fig(fig, "graph_coverage.png")

print("\nĐọc kèm:")
print("- Nếu mean|Δp| ~ 0 và ΔAUC ~ 0: mô hình đã học không dựa vào cạnh đồ thị.")
print(
    "- Nếu phần lớn dự đoán có out-degree = 0: GKT chỉ còn self-update, gần với Zero graph."
)

## 7. Case study: 5 dự đoán đúng + 5 dự đoán sai kèm bằng chứng

Khác bản cũ (chọn 5 case confidence cao nhất), ở đây chọn **theo tầng confidence** để có cả case
chắc chắn lẫn case sát ngưỡng, đồng thời ưu tiên 5 khái niệm khác nhau. Mỗi case ghi:

1. Lịch sử 8 bước trước dự đoán và tỉ lệ đúng cùng kỹ năng.
2. Láng giềng mạnh nhất trên đồ thị Transition (train only).
3. **Phản chứng**: lật kết quả một lần làm bài trong quá khứ → `Δp`.
4. **Ablation đồ thị tại chính case**: `A := 0` → `Δp`.
5. **Occlusion**: bỏ hẳn một bước (feature và question = padding) → `Δp`.
6. **Trạng thái tri thức** `h_t` tại khái niệm đích: norm, hạng trong toàn bộ khái niệm, top-5
   khái niệm theo norm, norm của các láng giềng đồ thị.

Lưu ý: norm `h_t` là **proxy** trạng thái ẩn, không phải mastery đã hiệu chỉnh. Báo cáo phải ghi rõ.


In [ ]:
CONF_EDGES = [0.4, 0.3, 0.2, 0.1, 0.0]


def _conf_bin(conf):
    for i, edge in enumerate(CONF_EDGES):
        if conf >= edge:
            return i
    return len(CONF_EDGES) - 1


def _skill_name(lookup, skill_id):
    if lookup and skill_id in lookup:
        return f"{skill_id}:{lookup[skill_id]}"
    return str(skill_id)


def _top_neighbors(graph, skill_id, topk=5):
    if graph is None:
        return []
    row = graph[skill_id].detach().cpu()
    values, idx = torch.topk(row, k=min(topk, row.numel()))
    return [
        (int(i), float(v))
        for v, i in zip(values.tolist(), idx.tolist())
        if i != skill_id and v > 0
    ]


def gather_candidates(model, loader, device):
    model.eval()
    uids = user_ids_from_loader(loader)
    base = loader.dataset.dataset
    candidates = {"correct": [], "incorrect": []}
    offset = 0
    with torch.no_grad():
        for batch in loader:
            features, questions, answers = [x.to(device) for x in batch]
            out = model(features, questions)
            pred = out[0] if isinstance(out, tuple) else out
            real = answers[:, 1:]
            mask = (real != -1) & (questions[:, 1:] != -1)
            bsz, tlen = pred.shape
            for b in range(bsz):
                pos = offset + b
                for t in range(tlen):
                    if not bool(mask[b, t]):
                        continue
                    p = float(pred[b, t].cpu())
                    y = int(real[b, t].cpu())
                    yhat = int(p >= 0.5)
                    candidates["correct" if yhat == y else "incorrect"].append(
                        {
                            "pos": pos,
                            "user_id": uids[pos] if pos < len(uids) else None,
                            "t": t + 1,
                            "idx": t,
                            "skill": int(questions[b, t + 1].cpu()),
                            "y_true": y,
                            "y_prob": p,
                            "y_pred": yhat,
                            "confidence": abs(p - 0.5),
                        }
                    )
            offset += bsz
    return candidates


def select_stratified(cands, n_each=5):
    if len(cands) <= n_each:
        return list(cands)
    picked, used = [], set()
    bin_order = [0, 3, 1, 4, 2]
    for b in bin_order:
        pool = [c for c in cands if _conf_bin(c["confidence"]) == b]
        pool.sort(key=lambda c: c["confidence"], reverse=True)
        for c in pool:
            if c["skill"] in used:
                continue
            picked.append(c)
            used.add(c["skill"])
            break
        if len(picked) >= n_each:
            break
    if len(picked) < n_each:
        for c in sorted(cands, key=lambda c: c["confidence"], reverse=True):
            if c["skill"] in used:
                continue
            picked.append(c)
            used.add(c["skill"])
            if len(picked) >= n_each:
                break
    if len(picked) < n_each:
        chosen = {id(c) for c in picked}
        for c in sorted(cands, key=lambda c: c["confidence"], reverse=True):
            if id(c) in chosen:
                continue
            picked.append(c)
            if len(picked) >= n_each:
                break
    return picked[:n_each]


def build_case_records(selected, loader, graph, lookup, history_k=8):
    ds = loader.dataset
    base = ds.dataset
    indices = ds.indices
    records = []
    for c in selected:
        i = indices[c["pos"]]
        qs = base.questions[i]
        ans = base.answers[i]
        idx = c["idx"]
        history = []
        for h in range(max(0, idx + 1 - history_k), idx + 1):
            sid = qs[h]
            if sid < 0:
                continue
            history.append(
                {
                    "t": h,
                    "skill": int(sid),
                    "skill_name": _skill_name(lookup, int(sid)),
                    "correct": int(ans[h]),
                }
            )
        records.append(
            {
                "pos": c["pos"],
                "user_id": c["user_id"],
                "t": c["t"],
                "skill": c["skill"],
                "skill_name": _skill_name(lookup, c["skill"]),
                "y_true": c["y_true"],
                "y_prob": c["y_prob"],
                "y_pred": c["y_pred"],
                "confidence": c["confidence"],
                "history": history,
                "neighbors": _top_neighbors(graph, c["skill"]),
            }
        )
    return records


def _forward_pred(model, features, questions, idx):
    out = model(features, questions)
    pred = out[0] if isinstance(out, tuple) else out
    return float(pred[0, idx])


def enrich_cases(model, loader, buckets, lookup, device, max_hist=8):
    ds = loader.dataset
    base = ds.dataset
    indices = ds.indices
    for rec in [r for bucket in buckets.values() for r in bucket]:
        i = indices[rec["pos"]]
        feats = base.features[i]
        qs = base.questions[i]
        idx = rec["t"] - 1
        f = torch.LongTensor([feats]).to(device)
        q = torch.LongTensor([qs]).to(device)
        ev = rec.setdefault("evidence", {})
        occ = []
        for h in rec["history"][-max_hist:]:
            tf, tq = f.clone(), q.clone()
            tf[0, h["t"]] = -1
            tq[0, h["t"]] = -1
            try:
                p_occ = _forward_pred(model, tf, tq, idx)
            except Exception:
                continue
            occ.append(
                {
                    "t": h["t"],
                    "skill": h["skill"],
                    "skill_name": h["skill_name"],
                    "was_correct": h["correct"],
                    "p": p_occ,
                    "delta": p_occ - rec["y_prob"],
                }
            )
        occ.sort(key=lambda r: abs(r["delta"]), reverse=True)
        ev["occlusion"] = occ
        try:
            out = model(f, q, return_state=True)
            states = out[4]
            norms = states[idx][0].norm(dim=-1).detach().cpu().numpy()
            target_norm = float(norms[rec["skill"]])
            order = np.argsort(norms)[::-1]
            ev["state"] = {
                "target_norm": target_norm,
                "target_rank": int((norms > target_norm).sum() + 1),
                "n_concepts": int(len(norms)),
                "global_mean_norm": float(norms.mean()),
                "top_concepts": [
                    {
                        "skill": int(j),
                        "skill_name": _skill_name(lookup, int(j)),
                        "norm": float(norms[j]),
                    }
                    for j in order[:5]
                ],
                "neighbor_norms": [
                    {
                        "skill": int(sid),
                        "skill_name": _skill_name(lookup, int(sid)),
                        "norm": float(norms[sid]),
                    }
                    for sid, _ in rec["neighbors"][:5]
                ],
            }
        except Exception as exc:
            ev["state_error"] = f"{type(exc).__name__}: {exc}"
    return buckets

In [ ]:
gkt_res = seed_models["GKT-Transition"]
gkt_model = gkt_res["model"].to(device)
gkt_bundle = gkt_res["bundle"]
transition_graph_cpu = gkt_bundle["graph"].detach().cpu()
skill_lookup_case = skill_lookup_from_loader(gkt_bundle["test_loader"])

candidates = gather_candidates(gkt_model, gkt_bundle["test_loader"], device)
print("Số ứng viên:", {k: len(v) for k, v in candidates.items()})

buckets = {
    key: build_case_records(
        select_stratified(candidates[key], n_each=5),
        gkt_bundle["test_loader"],
        transition_graph_cpu,
        skill_lookup_case,
    )
    for key in ("correct", "incorrect")
}
print("Đã chọn:", {k: len(v) for k, v in buckets.items()})
for key, rows in buckets.items():
    for rec in rows:
        print(
            f"  [{key}] user={rec['user_id']} t={rec['t']} skill={rec['skill_name']} "
            f"y={rec['y_true']} p={rec['y_prob']:.3f} conf={rec['confidence']:.3f}"
        )

try:
    annotate_evidence(
        gkt_model,
        gkt_bundle["test_loader"],
        buckets,
        graph=transition_graph_cpu,
        device=device,
    )
except Exception as exc:
    print(f"[cảnh báo] annotate_evidence lỗi: {type(exc).__name__}: {exc}")

buckets = enrich_cases(
    gkt_model, gkt_bundle["test_loader"], buckets, skill_lookup_case, device
)
write_json(buckets, SAVE_DIR / "cases.json")

case_rows = []
for key, rows in buckets.items():
    for rank, rec in enumerate(rows, 1):
        ev = rec.get("evidence") or {}
        flips = ev.get("flips") or []
        occ = ev.get("occlusion") or []
        case_rows.append(
            {
                "bucket": key,
                "rank": rank,
                "user_id": rec["user_id"],
                "t": rec["t"],
                "skill_name": rec["skill_name"],
                "y_true": rec["y_true"],
                "y_prob": rec["y_prob"],
                "y_pred": rec["y_pred"],
                "confidence": rec["confidence"],
                "evidence_ok": bool(ev.get("ok", False)),
                "max_flip_abs_dp": max(
                    (abs(f["delta"]) for f in flips), default=float("nan")
                ),
                "graph_dp": ev.get("delta_graph", float("nan")),
                "max_occ_abs_dp": max(
                    (abs(o["delta"]) for o in occ), default=float("nan")
                ),
                "target_state_norm": (ev.get("state") or {}).get(
                    "target_norm", float("nan")
                ),
                "target_state_rank": (ev.get("state") or {}).get(
                    "target_rank", float("nan")
                ),
            }
        )
case_summary_df = pd.DataFrame(case_rows)
write_csv(case_summary_df, "case_evidence_summary.csv")
print(case_summary_df.to_string(index=False))

In [ ]:
def why_text(rec):
    ev = rec.get("evidence") or {}
    if ev.get("ok"):
        parts = []
        flips = ev.get("flips") or []
        if flips:
            s = flips[0]
            if abs(s["delta"]) >= 1e-3:
                direction = "tăng" if s["delta"] > 0 else "giảm"
                parts.append(
                    f"bước ảnh hưởng mạnh nhất là t{s['t']} ({s['skill_name']}): "
                    f"lật kết quả làm p {direction} {abs(s['delta']):.3f}"
                )
        if "delta_graph" in ev:
            d = ev["delta_graph"]
            if abs(d) < 1e-3:
                parts.append("bỏ toàn bộ cạnh đồ thị không đổi p (|Δp| < 0.001)")
            else:
                direction = "tăng" if d > 0 else "giảm"
                parts.append(
                    f"đồ thị làm p {direction} {abs(d):.3f} "
                    f"(p={rec['y_prob']:.3f} vs {ev['nograph_p']:.3f} khi A := 0)"
                )
        occ = ev.get("occlusion") or []
        if occ and abs(occ[0]["delta"]) >= 1e-3:
            s = occ[0]
            parts.append(
                f"bỏ hẳn lần t{s['t']} ({s['skill_name']}) làm p thay đổi {s['delta']:+.3f}"
            )
        same = [h for h in rec["history"] if h["skill"] == rec["skill"]]
        if same:
            rate = sum(h["correct"] for h in same) / len(same)
            parts.append(f"lịch sử cùng kỹ năng: {rate:.2f} đúng trên {len(same)} lần")
        if parts:
            head = "Dự đoán đúng:" if rec["y_pred"] == rec["y_true"] else "Dự đoán sai:"
            return head + " " + "; ".join(parts) + "."

    same = [h for h in rec["history"] if h["skill"] == rec["skill"]]
    if rec["y_pred"] == rec["y_true"] and rec["y_true"] == 1:
        if same and sum(h["correct"] for h in same) >= 1:
            return (
                "Dự đoán đúng: lịch sử đã có câu đúng cùng kỹ năng nên mô hình nâng p."
            )
        return "Dự đoán đúng: các bước gần đây thành công nên mô hình ngoại suy sang kỹ năng này."
    if rec["y_pred"] == rec["y_true"] and rec["y_true"] == 0:
        return "Dự đoán đúng: lỗi trước đó ở kỹ năng cùng/liên quan giữ mastery thấp."
    if rec["y_pred"] == 1 and rec["y_true"] == 0:
        return (
            "Dương tính giả: mô hình thấy các thành công gần đây và ngoại suy quá mức."
        )
    return (
        f"Âm tính giả: nhãn thực tế đúng nhưng p={rec['y_prob']:.3f}; "
        "bằng chứng cùng kỹ năng thưa hoặc láng giềng xung đột."
    )


def format_case_report(buckets, graph_note="Transition (train only)"):
    lines = []
    for label, rows in buckets.items():
        lines.append(f"## {len(rows)} {label} predictions")
        for i, rec in enumerate(rows, 1):
            lines.append(
                f"\n### {label} #{i}  user={rec.get('user_id')}  t={rec.get('t')}  "
                f"skill={rec['skill_name']}  y={rec['y_true']}  p={rec['y_prob']:.3f}  "
                f"pred={rec['y_pred']}  confidence={rec['confidence']:.3f}"
            )
            hist_txt = ", ".join(
                f"t{h['t']}:{h['skill_name']}={'OK' if h['correct'] == 1 else 'NG'}"
                for h in rec["history"]
            )
            lines.append(f"- history ({len(rec['history'])} steps): {hist_txt}")
            same = [h for h in rec["history"] if h["skill"] == rec["skill"]]
            if same:
                rate = sum(h["correct"] for h in same) / len(same)
                lines.append(
                    f"- same-skill history: {len(same)} tries, accuracy={rate:.2f}"
                )
            else:
                lines.append("- same-skill history: none (cold skill for this student)")
            if rec["neighbors"]:
                neigh = ", ".join(f"{sid}:{w:.3f}" for sid, w in rec["neighbors"][:5])
                lines.append(f"- graph neighbors ({graph_note}): {neigh}")

            ev = rec.get("evidence") or {}
            if not ev.get("ok"):
                lines.append(
                    f"- counterfactual: not available ({ev.get('reason', 'not computed')})"
                )
            else:
                flips = ev.get("flips") or []
                if flips:
                    top = flips[0]
                    lines.append(
                        f"- counterfactual (flip one past attempt): t{top['t']} "
                        f"{top['skill_name']} ({'OK' if top['was_correct'] else 'NG'} -> "
                        f"{'NG' if top['was_correct'] else 'OK'}): p {rec['y_prob']:.3f} -> "
                        f"{top['p']:.3f} (dp={top['delta']:+.3f})"
                    )
                    others = "; ".join(
                        f"t{f['t']} {f['skill_name']} dp={f['delta']:+.3f}"
                        for f in flips[1:3]
                    )
                    if others:
                        lines.append(f"  - next largest: {others}")
                if "delta_graph" in ev:
                    lines.append(
                        f"- graph ablation for THIS prediction: A := 0 moves p "
                        f"{rec['y_prob']:.3f} -> {ev['nograph_p']:.3f} "
                        f"(dp={ev['nograph_p'] - rec['y_prob']:+.3f})"
                    )
                occ = ev.get("occlusion") or []
                if occ:
                    top = occ[0]
                    lines.append(
                        f"- occlusion (remove one past attempt): t{top['t']} "
                        f"{top['skill_name']} removed -> p {rec['y_prob']:.3f} -> "
                        f"{top['p']:.3f} (dp={top['delta']:+.3f})"
                    )
                    others = "; ".join(
                        f"t{o['t']} {o['skill_name']} dp={o['delta']:+.3f}"
                        for o in occ[1:3]
                    )
                    if others:
                        lines.append(f"  - next largest: {others}")
                edges = ev.get("edges") or []
                if edges:
                    txt = ", ".join(
                        f"{e['skill']} w={e['weight']:.3f} {e['direction']}"
                        + (
                            f" (learner last touched t{e['last_t']}, "
                            f"{'OK' if e['last_correct'] else 'NG'})"
                            if e["last_t"] is not None
                            else " (never touched by this learner)"
                        )
                        for e in edges
                    )
                    lines.append(f"- strongest edges at the target concept: {txt}")

                st = ev.get("state")
                if st:
                    lines.append(
                        f"- knowledge state h_t at target concept: norm={st['target_norm']:.3f}, "
                        f"rank {st['target_rank']}/{st['n_concepts']} "
                        f"(global mean={st['global_mean_norm']:.3f})"
                    )
                    top_txt = ", ".join(
                        f"{x['skill_name']} ({x['norm']:.2f})"
                        for x in st["top_concepts"]
                    )
                    lines.append(f"  - top-5 concepts by state norm: {top_txt}")
                    if st["neighbor_norms"]:
                        nbr_txt = ", ".join(
                            f"{x['skill_name']} ({x['norm']:.2f})"
                            for x in st["neighbor_norms"]
                        )
                        lines.append(f"  - graph-neighbor state norms: {nbr_txt}")
                elif "state_error" in ev:
                    lines.append(
                        f"- knowledge state: not available ({ev['state_error']})"
                    )

            lines.append(f"- why: {why_text(rec)}")
    return "\n".join(lines)


case_report = format_case_report(buckets)
(SAVE_DIR / "cases.md").write_text(case_report, encoding="utf-8")
print(case_report)
print("\n[OK] Đã lưu", SAVE_DIR / "cases.md")

fig, ax = plt.subplots(figsize=(11, 4.2))
labels, flip_vals, occ_vals, graph_vals = [], [], [], []
for key, rows in buckets.items():
    for rec in rows:
        ev = rec.get("evidence") or {}
        labels.append(f"{key[:1].upper()}:{rec['user_id']}@{rec['t']}")
        flips = ev.get("flips") or []
        occ = ev.get("occlusion") or []
        flip_vals.append(max((abs(f["delta"]) for f in flips), default=0.0))
        occ_vals.append(max((abs(o["delta"]) for o in occ), default=0.0))
        graph_vals.append(
            abs(ev.get("delta_graph", 0.0)) if "delta_graph" in ev else 0.0
        )
x = np.arange(len(labels))
w = 0.27
ax.bar(x - w, flip_vals, w, label="|Δp| lật 1 bước")
ax.bar(x, occ_vals, w, label="|Δp| bỏ 1 bước")
ax.bar(x + w, graph_vals, w, label="|Δp| bỏ đồ thị")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("|Δp|")
ax.set_title("Bằng chứng định lượng cho 10 case (seed 42)")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "case_evidence.png")

## 8. Dashboard chọn người học (Nhiệm vụ §8)

`export_learner_view` chạy GKT-Transition trên test set, gom theo người học:

- xác suất dự đoán từng bước,
- lịch sử làm bài,
- trạng thái tri thức ước lượng: norm `h_t` của từng khái niệm tại từng bước.

JSON được giới hạn `DASHBOARD_MAX_LEARNERS` để không phình lên hàng trăm MB; bảng tóm tắt và hình
một người học được lưu riêng để dán vào báo cáo.


In [ ]:
if RUN_DASHBOARD:
    gkt_res = seed_models["GKT-Transition"]
    dash_model = gkt_res["model"].to(device)
    dash_bundle = gkt_res["bundle"]
    dash_lookup = skill_lookup_from_loader(dash_bundle["test_loader"])
    dash_graph = (
        dash_bundle["graph"].detach().cpu()
        if dash_bundle.get("graph") is not None
        else None
    )

    views = export_learner_view(
        dash_model,
        dash_bundle["test_loader"],
        skill_lookup=dash_lookup,
        graph=dash_graph,
        device=device,
        model_type="GKT",
        out_path=SAVE_DIR / "learner_view.json",
        max_learners=DASHBOARD_MAX_LEARNERS,
    )
    print(f"Đã export {len(views)} người học -> {SAVE_DIR / 'learner_view.json'}")

    rows = []
    for view in views:
        steps = view["steps"]
        if not steps:
            continue
        correct = [s["y_pred"] == s["y_true"] for s in steps]
        counts = Counter(s["skill"] for s in steps)
        dominant = counts.most_common(1)[0][0]
        dom_norms = [
            s["state_norm"][dominant]
            for s in steps
            if "state_norm" in s and dominant < len(s["state_norm"])
        ]
        last_norm = (
            float(np.mean(steps[-1]["state_norm"]))
            if "state_norm" in steps[-1]
            else float("nan")
        )
        rows.append(
            {
                "user_id": view["user_id"],
                "n_steps": len(steps),
                "accuracy": float(np.mean(correct)),
                "mean_p": float(np.mean([s["y_prob"] for s in steps])),
                "dominant_skill": dominant,
                "dominant_skill_name": _skill_name(dash_lookup, dominant),
                "mean_dominant_state_norm": (
                    float(np.mean(dom_norms)) if dom_norms else float("nan")
                ),
                "final_mean_state_norm": last_norm,
            }
        )
    learner_df = pd.DataFrame(rows)
    write_csv(learner_df, "learner_view_summary.csv")
    print(learner_df.head(10).to_string(index=False))

    if views:
        view = views[0]
        steps = view["steps"]
        ts = [s["t"] for s in steps]
        ps = [s["y_prob"] for s in steps]
        ys = [s["y_true"] for s in steps]
        counts = Counter(s["skill"] for s in steps)
        dominant = counts.most_common(1)[0][0]
        dom_norms = [
            s["state_norm"][dominant]
            for s in steps
            if "state_norm" in s and dominant < len(s["state_norm"])
        ]
        fig, ax = plt.subplots(figsize=(11, 3.8))
        ax.plot(ts, ps, color="tab:blue", linewidth=1.2, label="p̂ đúng")
        ax.scatter(
            [t for t, y in zip(ts, ys) if y == 1],
            [p for p, y in zip(ps, ys) if y == 1],
            marker="o",
            s=18,
            color="green",
            label="thực tế đúng",
            zorder=3,
        )
        ax.scatter(
            [t for t, y in zip(ts, ys) if y == 0],
            [p for p, y in zip(ps, ys) if y == 0],
            marker="x",
            s=22,
            color="red",
            label="thực tế sai",
            zorder=3,
        )
        ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
        ax.set_ylim(0, 1)
        ax.set_xlabel("bước tương tác")
        ax.set_ylabel("xác suất dự đoán đúng")
        ax.set_title(
            f"Người học {view['user_id']} — {len(steps)} bước "
            f"(kỹ năng chính: {_skill_name(dash_lookup, dominant)})"
        )
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc="lower left")
        if dom_norms and len(dom_norms) == len(ts):
            ax2 = ax.twinx()
            ax2.plot(
                ts,
                dom_norms,
                color="tab:orange",
                linestyle=":",
                linewidth=1.2,
                label=f"norm h_t của kỹ năng {dominant}",
            )
            ax2.set_ylabel("norm trạng thái tri thức")
            ax2.legend(fontsize=8, loc="upper right")
        fig.tight_layout()
        save_fig(fig, f"learner_{view['user_id']}.png")
        (SAVE_DIR / "learner_example.txt").write_text(
            render_learner_view(views, 0), encoding="utf-8"
        )
        print(render_learner_view(views, 0))

    interactive_learner_view(views)

## 9. Manifest tái lập và báo cáo tổng hợp tự động

Cell này ghi:

- `run_manifest.json`: commit Git, hash dữ liệu, cấu hình, seed, split, thông số đồ thị, phiên bản gói.
- `environment.txt`: hệ điều hành, Python, phiên bản thư viện, GPU.
- `report_summary.md`: **toàn bộ bảng + kết luận trung thực + danh sách hình**, viết sẵn để dán vào
  báo cáo 12–18 trang. Số liệu được đọc lại từ đĩa nên cell chạy độc lập được sau khi train xong.


In [ ]:
env_lines = [
    f"platform: {platform.platform()}",
    f"python: {sys.version}",
    f"torch: {torch.__version__}",
    f"numpy: {np.__version__}",
    f"pandas: {pd.__version__}",
    f"scipy: {scipy.__version__}",
    f"scikit-learn: {sklearn.__version__}",
    f"matplotlib: {matplotlib.__version__}",
    f"device: {device}",
    f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}",
    f"git_commit: {GIT_COMMIT}",
    f"notebook: {NOTEBOOK_VERSION}",
]
(SAVE_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")

manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "git_commit": GIT_COMMIT,
    "generated_utc": utc_now(),
    "data": DATA_STATS,
    "config": {
        "quick_run": QUICK_RUN,
        "data_file": DATA_FILE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_seq_len": MAX_SEQ_LEN,
        "seeds": SEEDS,
        "graph_types": GRAPH_TYPES,
        "run_dashboard": RUN_DASHBOARD,
        "dashboard_max_learners": DASHBOARD_MAX_LEARNERS,
    },
    "packages": {
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit-learn": sklearn.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "split": split_df.to_dict(orient="records"),
    "graphs": graph_stats_df.to_dict(orient="records"),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
write_json(manifest, SAVE_DIR / "run_manifest.json")


ms = ms_table.copy()
paired = paired_df.copy()
calib = calib_df.copy()
ginf = graph_inf_df.copy()
coverage = coverage_df.copy()
gstats = graph_stats_df.copy()
split_tbl = split_df.copy()
case_tbl = case_summary_df.copy()

best = ms.sort_values("auc_mean", ascending=False).iloc[0]
by_name = ms.set_index("model")
dkt = by_name.loc["DKT"]
trans = by_name.loc["GKT-Transition"]
zero = by_name.loc["GKT-Zero"]
shuf = by_name.loc["GKT-Shuffled"]
shufrow = by_name.loc["GKT-ShuffledRow"]
dense = by_name.loc["GKT-Dense"]


def _pair(a, b, metric):
    hit = paired[(paired["comparison"] == f"{a} − {b}") & (paired["metric"] == metric)]
    return hit.iloc[0] if len(hit) else None


p_trans_zero = _pair("GKT-Transition", "GKT-Zero", "test_auc")
p_trans_shuf = _pair("GKT-Transition", "GKT-Shuffled", "test_auc")
p_trans_shufrow = _pair("GKT-Transition", "GKT-ShuffledRow", "test_auc")
p_dkt_trans = _pair("DKT", "GKT-Transition", "test_auc")

findings = []
findings.append(
    f"- Mô hình tốt nhất theo Test AUC trung bình: **{best['model']}** "
    f"({best['auc_mean']:.4f} ± {best['auc_std']:.4f})."
)
findings.append(
    f"- DKT (không đồ thị): {dkt['auc_mean']:.4f} ± {dkt['auc_std']:.4f}; "
    f"GKT-Transition: {trans['auc_mean']:.4f} ± {trans['auc_std']:.4f}."
)
if p_dkt_trans is not None:
    findings.append(
        f"- DKT − GKT-Transition: Δ={p_dkt_trans['mean_diff']:+.4f} "
        f"(CI95 [{p_dkt_trans['ci95_low']:+.4f}, {p_dkt_trans['ci95_high']:+.4f}], "
        f"p={p_dkt_trans['p']:.3f})."
    )
if p_trans_zero is not None:
    findings.append(
        f"- Transition − Zero: Δ={p_trans_zero['mean_diff']:+.4f} "
        f"(CI95 [{p_trans_zero['ci95_low']:+.4f}, {p_trans_zero['ci95_high']:+.4f}], "
        f"p={p_trans_zero['p']:.3f}) → "
        + (
            "có ý nghĩa thống kê."
            if p_trans_zero["significant_0.05"]
            else "chưa đủ bằng chứng."
        )
    )
if p_trans_shuf is not None:
    findings.append(
        f"- Transition − Shuffled: Δ={p_trans_shuf['mean_diff']:+.4f} "
        f"(CI95 [{p_trans_shuf['ci95_low']:+.4f}, {p_trans_shuf['ci95_high']:+.4f}], "
        f"p={p_trans_shuf['p']:.3f}) → "
        + (
            "cấu trúc cạnh có tác dụng."
            if p_trans_shuf["significant_0.05"]
            else "chưa chứng minh được vai trò cấu trúc cạnh."
        )
    )
if p_trans_shufrow is not None:
    findings.append(
        f"- Transition − ShuffledRow (giữ out-degree): Δ={p_trans_shufrow['mean_diff']:+.4f} "
        f"(CI95 [{p_trans_shufrow['ci95_low']:+.4f}, {p_trans_shufrow['ci95_high']:+.4f}], "
        f"p={p_trans_shufrow['p']:.3f})."
    )

cov_any = coverage.iloc[2]
findings.append(
    f"- Chỉ **{cov_any['Tỉ lệ']:.1%}** dự đoán test có khái niệm đích với ít nhất một cạnh "
    "trong đồ thị Transition train-only."
)
for _, r in ginf.iterrows():
    findings.append(
        f"- {r['model']}: bỏ đồ thị lúc suy luận ΔAUC={r['delta_auc']:+.4f}, "
        f"mean|Δp|={r['mean_abs_dp']:.4f}."
    )

pair_disp = paired.copy()
for c in [
    "mean_a",
    "mean_b",
    "mean_diff",
    "ci95_low",
    "ci95_high",
    "t",
    "p",
    "cohen_dz",
]:
    pair_disp[c] = pair_disp[c].astype(float).round(4)
pair_disp = pair_disp[
    [
        "comparison",
        "metric",
        "n_seeds",
        "mean_a",
        "mean_b",
        "mean_diff",
        "ci95_low",
        "ci95_high",
        "p",
        "cohen_dz",
        "significant_0.05",
    ]
]

calib_disp = calib.copy()
for c in [
    "auc",
    "nll",
    "brier",
    "ece",
    "ace",
    "mean_pred",
    "observed_rate",
    "over_prediction",
]:
    calib_disp[c] = calib_disp[c].astype(float).round(4)
calib_disp = calib_disp[
    [
        "model",
        "n",
        "auc",
        "nll",
        "brier",
        "ece",
        "ace",
        "mean_pred",
        "observed_rate",
        "over_prediction",
    ]
]

ginf_disp = ginf.copy()
for c in [
    "auc_with_graph",
    "auc_zero_graph",
    "delta_auc",
    "nll_with_graph",
    "nll_zero_graph",
    "delta_nll",
    "mean_abs_dp",
    "max_abs_dp",
]:
    ginf_disp[c] = ginf_disp[c].astype(float).round(4)

case_disp = case_tbl.copy()
for c in [
    "y_prob",
    "confidence",
    "max_flip_abs_dp",
    "graph_dp",
    "max_occ_abs_dp",
    "target_state_norm",
    "target_state_rank",
]:
    if c in case_disp.columns:
        case_disp[c] = case_disp[c].astype(float).round(4)

checklist_md = """
| Yêu cầu / GAP | Nơi đáp ứng trong notebook này |
|---|---|
| §5 Cấp 1: baseline không đồ thị + mô hình đồ thị | DKT + GKT (7 cấu hình) — §3 |
| §5 Cấp 1: ablation bỏ cạnh / ma trận đơn vị | Dense, Identity, Zero, Shuffled, ShuffledRow — §2, §4 |
| §6.2 đồ thị chỉ xây từ train | `leak_report.txt` đối chiếu support + khối lượng test-only — §2.2 |
| §6.3 cùng một split + seed tái lập | `split.json`, assert split trùng trên mọi bundle — §2, §2.1 |
| §6.4 tách theo người học | assert ba tập rời nhau — §2 |
| §7 AUC / Accuracy / NLL | bảng chính + đa seed — §4 |
| §7 mean ± std ≥ 3 seed | 5 seed, split cố định — §3, §4 |
| §7 vai trò đồ thị | paired t-test + bootstrap CI + `A:=0` lúc suy luận + độ phủ — §4, §6 |
| §7 mở rộng calibration | Brier / ECE / ACE / reliability — §5 |
| §8 dashboard chọn người học | `learner_view.json`, bảng tóm tắt, hình, dropdown — §8 |
| §8 5 đúng + 5 sai kèm bằng chứng | `cases.md` + `cases.json` + hình — §7 |
| GAP P0.1 đa seed có Dense, split cố định | §3 chạy cả 7 cấu hình × 5 seed |
| GAP P1.6 bằng chứng case | phản chứng, ablation đồ thị, occlusion, trạng thái `h_t` — §7 |
| GAP P1.8 Identity ≡ Zero, edge-shuffle | giải thích trong `README.md`; thêm `ShuffledRow` giữ out-degree — §2.1 |
| GAP E3 chưa chứng minh cấu trúc đồ thị | báo cáo trung thực bằng kiểm định + ablation suy luận — §4, §6 |
| GAP P0.3/P2.11 sản phẩm báo cáo | `report_summary.md` + 2 zip — §9, §10 |

**Còn phải làm thủ công:** viết báo cáo 12–18 trang và slide từ `report_summary.md`; Cấp độ 3
(quiz tự thu thập, 10 case đồ thị nâng cao) nằm ngoài phạm vi Cấp độ 1.
"""

report = f"""# Báo cáo tổng hợp Cấp độ 1 — sinh tự động

> Sinh bởi `{NOTEBOOK_VERSION}` lúc {utc_now()}. Số liệu lấy từ `tables/`, hình từ `figures/`.
> Chạy `QUICK_RUN=False` trên ASSISTments 2009 đầy đủ mới dùng cho báo cáo.

## 1. Metadata

- Git commit: `{GIT_COMMIT}`
- Dữ liệu: `{DATA_FILE}` | SHA-256 `{DATA_STATS['sha256']}`
- Kích thước: {DATA_STATS['size_mb']} MB | {DATA_STATS['rows']:,} dòng | {DATA_STATS['users']:,} người học | {DATA_STATS['skills']:,} khái niệm
- Cấu hình: epochs={EPOCHS}, batch={BATCH_SIZE}, max_seq_len={MAX_SEQ_LEN}, seeds={SEEDS}
- Thiết bị: `{device}`

## 2. Chia tập (theo người học, seed=42)

{md_table(split_tbl)}

## 3. Đồ thị

{md_table(gstats)}

Độ phủ đồ thị Transition theo dự đoán test:

{md_table(coverage)}

## 4. Kết quả chính (seed 42)

{md_table(main_display)}

## 5. Đa seed (mean ± std)

{md_table(ms_display)}

## 6. Kiểm định ghép cặp theo seed

{md_table(pair_disp)}

### Kết luận từ số liệu

{chr(10).join(findings)}

## 7. Hiệu chỉnh xác suất

{md_table(calib_disp)}

## 8. Đồ thị đóng góp lúc suy luận (A := 0)

{md_table(ginf_disp)}

## 9. Case study (5 đúng + 5 sai)

{md_table(case_disp)}

Chi tiết đầy đủ: `cases.md` và `cases.json`.

## 10. Hình cần dùng trong báo cáo

| Hình | Tệp |
|---|---|
| Phân phối độ dài chuỗi | `figures/eda_sequence_length.png` |
| Learning curves | `figures/learning_curves.png` |
| So sánh AUC/NLL (mean ± std) | `figures/model_comparison.png` |
| Reliability diagram | `figures/calibration_reliability.png` |
| Đồ thị đóng góp lúc suy luận | `figures/graph_inference_ablation.png` |
| Độ phủ đồ thị | `figures/graph_coverage.png` |
| Bằng chứng 10 case | `figures/case_evidence.png` |
| Một người học trong dashboard | `figures/learner_*.png` |

## 11. Đối chiếu yêu cầu

{checklist_md}

## 12. Lưu ý trung thực

- ASSISTments 2009 có **một khái niệm mỗi câu hỏi**, nên đồ thị Transition thưa; phần lớn dự đoán
  test có thể không có cạnh. Khi đó GKT gần với self-update và không khác Zero graph về bản chất.
- Nếu Shuffled ≈ Transition, kết luận đúng là **chưa chứng minh được vai trò cấu trúc cạnh** ở
  Cấp độ 1; lợi ích (nếu có) đến từ mật độ hàng xóm, không từ hình dạng đồ thị.
- Norm `h_t` là proxy trạng thái ẩn, **không phải mastery đã hiệu chỉnh**. Không dùng norm để kết
  luận người học "giỏi/yếu" tuyệt đối.
- `report_summary.md` chỉ chứa số liệu và diễn giải tự động; phần thảo luận, so sánh với bài báo
  và slide vẫn phải viết thủ công.
"""
(SAVE_DIR / "report_summary.md").write_text(report, encoding="utf-8")
print("[md]", SAVE_DIR / "report_summary.md")
print("\n=== KẾT LUẬN TỰ ĐỘNG ===")
print("\n".join(findings))

## 10. Đóng gói toàn bộ output để tải về

Kaggle xoá `/kaggle/working` khi phiên kết thúc, nên phải tải zip **trước khi hết phiên**.

- `level1_output.zip`: toàn bộ artifact — bảng, hình, `cases.md/json`, `learner_view.json`,
  `split.json`, `leak_report.txt`, `run_manifest.json`, `report_summary.md`, checkpoint seed 42.
- `level1_report_pack.zip`: bản nhẹ chỉ gồm bảng + hình + báo cáo + case + manifest, để gửi nhóm
  hoặc đưa vào phụ lục báo cáo.

Chạy lại cell này bất kỳ lúc nào để gói bổ sung; zip cũ bị ghi đè.


In [ ]:
readme = f"""level1_out — sản phẩm Cấp độ 1
Sinh bởi: {NOTEBOOK_VERSION}
Git commit: {GIT_COMMIT}
Dữ liệu: {DATA_FILE} (sha256={DATA_STATS['sha256']})
Cấu hình: epochs={EPOCHS}, batch={BATCH_SIZE}, max_seq_len={MAX_SEQ_LEN}, seeds={SEEDS}

Cấu trúc:
  tables/                  bảng số liệu CSV (UTF-8 BOM, mở được bằng Excel)
  figures/                 hình PNG 150 dpi để dán vào báo cáo
  cases.md / cases.json    10 case study + bằng chứng định lượng
  learner_view.json        dashboard người học (giới hạn {DASHBOARD_MAX_LEARNERS} người)
  learner_view_summary.csv bảng tóm tắt người học
  learner_example.txt      ví dụ render một người học
  split.json               split theo người học seed 42 (tái lập)
  leak_report.txt          kiểm tra chống rò rỉ
  run_manifest.json        manifest tái lập
  environment.txt          phiên bản môi trường
  report_summary.md        báo cáo tổng hợp tự động
  checkpoints/             checkpoint seed 42 (DKT + 6 biến thể GKT)

Cách dùng cho báo cáo:
  1. Mở report_summary.md để lấy bảng và kết luận.
  2. Dùng figures/ cho phần trực quan.
  3. Dùng cases.md cho mục xAI 5 đúng + 5 sai.
  4. Trích số từ tables/ khi cần bảng chi tiết.

Lưu ý: bản zip này không thay thế báo cáo/slide; chỉ cung cấp số liệu và hình.
"""
(SAVE_DIR / "README_pack.txt").write_text(readme, encoding="utf-8")


def package_dir(save_dir, zip_path, max_mb=200, keep_ckpts=True):
    save_dir = Path(save_dir)
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    zip_real = zip_path.resolve()
    added, skipped = [], []
    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(save_dir.rglob("*")):
            if not p.is_file() or p.resolve() == zip_real:
                continue
            if p.suffix == ".tmp":
                continue
            mb = p.stat().st_size / 1024**2
            rel = str(p.relative_to(save_dir.parent))
            if not keep_ckpts and p.suffix in (".pt", ".npz"):
                skipped.append((rel, mb, "KEEP_CKPTS=False"))
                continue
            if mb > max_mb and p.suffix != ".pt":
                skipped.append((rel, mb, f"> MAX_MB={max_mb}"))
                continue
            zf.write(p, arcname=rel)
            added.append((rel, mb))
    with zipfile.ZipFile(zip_path) as check:
        bad = check.testzip()
    print(
        f">>> {zip_path} | {zip_path.stat().st_size / 1024**2:.1f} MB | "
        f"{len(added)} file | kiểm tra zip: {'OK' if bad is None else f'HỎNG tại {bad}'}"
    )
    groups = defaultdict(lambda: [0, 0.0])
    for rel, mb in added:
        parts = rel.split("/")
        key = "/".join(parts[1:2]) if len(parts) > 2 else parts[-1]
        groups[key][0] += 1
        groups[key][1] += mb
    for key, (n, total) in sorted(groups.items()):
        print(f"    {key:34s} {n:4d} file {total:8.1f} MB")
    if skipped:
        print(f"    bỏ {len(skipped)} file:")
        for rel, mb, why in skipped:
            print(f"      {rel:52s} {mb:8.1f} MB | {why}")
    return zip_path


def package_selected(save_dir, zip_path, patterns):
    save_dir = Path(save_dir)
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    added = []
    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for pattern in patterns:
            for p in sorted(save_dir.glob(pattern)):
                if p.is_file():
                    zf.write(p, arcname=str(p.relative_to(save_dir.parent)))
                    added.append(p)
    with zipfile.ZipFile(zip_path) as check:
        bad = check.testzip()
    print(
        f">>> {zip_path} | {zip_path.stat().st_size / 1024**2:.1f} MB | "
        f"{len(added)} file | kiểm tra zip: {'OK' if bad is None else f'HỎNG tại {bad}'}"
    )
    return zip_path


package_dir(SAVE_DIR, ZIP_PATH, max_mb=200, keep_ckpts=True)
package_selected(
    SAVE_DIR,
    REPORT_ZIP_PATH,
    [
        "report_summary.md",
        "run_manifest.json",
        "environment.txt",
        "leak_report.txt",
        "split.json",
        "cases.md",
        "cases.json",
        "README_pack.txt",
        "tables/*",
        "figures/*",
    ],
)
print("\nTẢI VỀ TRƯỚC KHI PHIÊN KẾT THÚC:")
print(f"  - Toàn bộ: {ZIP_PATH}")
print(f"  - Bản nhẹ cho báo cáo: {REPORT_ZIP_PATH}")
print("  Kaggle: tab Output (hoặc sidebar Files) -> chọn zip -> tải xuống.")

## 11. Checklist đối chiếu và việc còn lại

### Đã giải quyết trong notebook này

- [x] Baseline không đồ thị **DKT** và mô hình đồ thị **GKT** trên cùng split.
- [x] Ablation **Dense / Identity / Zero / Shuffled / ShuffledRow**; giải thích `Identity ≡ Zero`.
- [x] Đa seed **5 seed**, split cố định, có cả `Dense`; lưu từng seed.
- [x] Kiểm định **paired t-test + bootstrap CI + Cohen's d_z**.
- [x] Báo cáo chống rò rỉ: split rời nhau, graph support khớp train, đo mass test-only.
- [x] Đo **đồ thị đóng góp lúc suy luận** (`A:=0`) và **độ phủ đồ thị** — trả lời trực tiếp E3.
- [x] Calibration: Brier, ECE, ACE, reliability diagram.
- [x] Case study 5 đúng + 5 sai, chọn theo tầng confidence, kèm phản chứng, ablation đồ thị,
      occlusion, trạng thái `h_t`, láng giềng.
- [x] Dashboard người học + bảng tóm tắt + hình.
- [x] Manifest tái lập, environment, `report_summary.md`.
- [x] Đóng gói `level1_output.zip` và `level1_report_pack.zip`.

### Việc còn lại (không chặn Cấp độ 1)

1. Viết báo cáo 12–18 trang + slide từ `report_summary.md`; thêm phần thảo luận so với DKT/GKT gốc.
2. Nếu muốn kết luận mạnh hơn về cấu trúc đồ thị: chạy thêm trên một tập có **nhiều khái niệm mỗi
   câu hỏi** (Junyi/Eedi) hoặc dùng tiêu chí quỹ đạo của Cấp độ 2 (`consistency_spread`).
3. Cấp độ 3: quiz tự thu thập + 10 case bằng chứng đồ thị nâng cao (ngoài phạm vi Cấp độ 1).
4. `cases.json` và `learner_view.json` là JSON lớn; nếu nộp qua Git, cân nhắc `.gitattributes`
   hoặc chỉ nộp zip.
